# exp096_projection_fadein_after_prefix inference

Fade-in projection inference port using the train-side selected variant.


## Contents

1. Setup and configuration
2. Selection contract
3. Run inference port
4. Preview outputs
5. Metrics and submission summary


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from projection_fadein_after_prefix import run_inference_from_config, to_jsonable

paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()
config.setdefault("runtime", {})["output_dir"] = str(paths.artifacts_dir)
config.setdefault("data", {})["raw_dir"] = str(paths.raw_data_dir)
config.setdefault("data", {})["train_dir"] = str(paths.train_data_dir)
config.setdefault("data", {})["test_dir"] = str(paths.test_data_dir)
config.setdefault("data", {})["sample_submission"] = str(paths.sample_submission_path)

print(json.dumps({
    "experiment": EXPERIMENT_NAME,
    "route": get_nested(config, "experiment.route"),
    "parent": get_nested(config, "lineage.parent"),
    "inference_mode": get_nested(config, "inference.mode"),
    "selected_variant": get_nested(config, "inference.selected_variant"),
    "test_dir": get_nested(config, "data.test_dir"),
    "artifacts_dir": str(paths.artifacts_dir),
    "submission_path": str(paths.submission_path),
}, indent=2, sort_keys=True))


## 2. Selection contract


In [ ]:
print(json.dumps({
    "selected_variant": get_nested(config, "inference.selected_variant"),
    "exp073_inference_predictions": get_nested(config, "data.exp073_inference_predictions"),
    "test_dir": get_nested(config, "data.test_dir"),
    "notes": get_nested(config, "inference.notes"),
}, indent=2, ensure_ascii=False))

for value in get_nested(config, "data.exp073_inference_predictions"):
    path = Path(value)
    print(value, "exists=" + str(path.exists()), "size=" + str(path.stat().st_size if path.exists() else 0))


## 3. Run inference port


In [ ]:
summary = run_inference_from_config(config)
print(json.dumps(to_jsonable(summary), indent=2, sort_keys=True))


## 4. Preview outputs


In [ ]:
for name, filename in summary.get("outputs", {}).items():
    path = paths.artifacts_dir / filename
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")
    if path.exists() and path.suffix in {".csv", ".gz"}:
        display(pd.read_csv(path, nrows=10))

if summary["status"] == "inference_projection_written":
    submission_path = Path(summary["submission"]["path"])
    print("submission", submission_path, submission_path.exists(), submission_path.stat().st_size if submission_path.exists() else 0)
    display(pd.read_csv(submission_path, nrows=10))


## 5. Metrics and submission summary


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "updated_at": datetime.now(UTC).isoformat(),
    "route": get_nested(config, "experiment.route"),
    "parent": get_nested(config, "lineage.parent"),
    "inference": summary,
}
metrics_path = paths.experiment_dir / "metrics.json"
with metrics_path.open("w") as fp:
    json.dump(to_jsonable(metrics), fp, indent=2, sort_keys=True)
print(json.dumps(to_jsonable(metrics), indent=2, sort_keys=True))
